# Train & Evaluate Recommendation Models

Notebook này sẽ xây dựng và đánh giá 4 thuật toán khuyến nghị phim dựa trên dữ liệu MovieLens đã được xử lý (rating + metadata).

Các thuật toán sẽ được triển khai: 
1. User-based Collaborative Filtering (UserCF)
2. Item-based Collaborative Filtering (ItemCF)
3. Matrix Factorization (SVD)
4. Content-Based Recommendation

Cuối cùng sẽ đánh giá bằng RMSE/MAE và Precision@K / Recall@K để so sánh hiệu năng.

# Phân tích dữ liệu trước khi train các model

Trước khi chọn và train các thuật toán, hãy hiểu đặc điểm dữ liệu:
- **Sparsity**: Phần trăm các entry rỗng trong user-item matrix
- **Density**: Phần trăm các entry có dữ liệu
- **Data Distribution**: Phân bố rating và sự thích hợp của các thuật toán

## 1) Load dữ liệu đã xử lý

Dữ liệu đã được tạo ra từ notebook xử lý dữ liệu (`Data_Processing.ipynb`). Ở đây chúng ta sẽ tải:
- `ratings_cleaning.csv`: rating đã làm sạch + timestamp
- `movies_metadata.csv`: metadata phục vụ gợi ý (genre, tag, era, v.v.)
- `model_features.csv`: feature numeric + one-hot + tfidf dùng cho content-based.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load dữ liệu đã xử lý
# The processed ratings file is expected to be in the workspace.
# If you don't have `ratings_cleaning.csv`, fallback to the original `ratings.csv`.
ratings_path = 'data/cleaning/ratings_cleaning.parquet'
metadata_path = 'data/cleaning/movies_metadata.parquet'
features_path = 'data/cleaning/model_features.parquet'

assert os.path.exists(ratings_path), f'File not found: {ratings_path}'
assert os.path.exists(metadata_path), f'File not found: {metadata_path}'
assert os.path.exists(features_path), f'File not found: {features_path}'

# Load preprocessed datasets
# Note: `movies_metadata.csv` and `model_features.csv` are assumed to be generated from the data processing pipeline.
df_ratings = pd.read_parquet(ratings_path)
df_metadata = pd.read_parquet(metadata_path)
df_features = pd.read_parquet(features_path)

print('ratings:', df_ratings.shape)
print('metadata:', df_metadata.shape)
print('features:', df_features.shape)

df_ratings.head()

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Phân tích chi tiết về dữ liệu
print("=" * 60)
print("PHÂN TÍCH CHI TIẾT DỮ LIỆU TRƯỚC TRAIN")
print("=" * 60)

# Thống kê cơ bản
n_users = df_ratings['userId'].nunique()
n_movies = df_ratings['movieId'].nunique()
n_ratings = len(df_ratings)

# Tính độ thưa/mật độ
total_possible_ratings = n_users * n_movies
sparsity = 1 - (n_ratings / total_possible_ratings)
density = n_ratings / total_possible_ratings

print(f"\n1. THỐNG KÊ CƠ BẢN:")
print(f"   - Số users: {n_users:,}")
print(f"   - Số movies: {n_movies:,}")
print(f"   - Số ratings: {n_ratings:,}")
print(f"   - Tổng có thể ratings: {total_possible_ratings:,}")
print(f"   - Sparsity (độ thưa): {sparsity:.4f} ({sparsity*100:.2f}%)")
print(f"   - Density (độ mật): {density:.4f} ({density*100:.2f}%)")

# Thống kê rating
print(f"\n2. PHÂN BỐ RATING:")
print(f"   - Min rating: {df_ratings['rating'].min()}")
print(f"   - Max rating: {df_ratings['rating'].max()}")
print(f"   - Mean rating: {df_ratings['rating'].mean():.2f}")
print(f"   - Std rating: {df_ratings['rating'].std():.2f}")
print(f"\n   Hệ thống rating:")
print(df_ratings['rating'].value_counts().sort_index())

# Thống kê user
user_activity = df_ratings.groupby('userId').size()
print(f"\n3. HOẠT ĐỘNG CỦA USER:")
print(f"   - Ratings/user trung bình: {user_activity.mean():.2f}")
print(f"   - Min ratings/user: {user_activity.min()}")
print(f"   - Max ratings/user: {user_activity.max()}")
print(f"   - Median ratings/user: {user_activity.median():.0f}")

# Thống kê movie
movie_popularity = df_ratings.groupby('movieId').size()
print(f"\n4. PHỔ BIẾN CỦA MOVIE:")
print(f"   - Ratings/movie trung bình: {movie_popularity.mean():.2f}")
print(f"   - Min ratings/movie: {movie_popularity.min()}")
print(f"   - Max ratings/movie: {movie_popularity.max()}")
print(f"   - Median ratings/movie: {movie_popularity.median():.0f}")

print(f"\n5. LỰA CHỌN THUẬT TOÁN DỰA VÀO DỮ LIỆU:")
print(f"   • Sparsity cao ({sparsity:.1%}) → Cần xử lý sparsity tốt")
print(f"   • User activity tương đối đều (avg={user_activity.mean():.1f}) → UserCF/ItemCF khả thi")
print(f"   • Movie popularity phân bố lệch (median={movie_popularity.median():.0f}, max={movie_popularity.max()}) → SVD sẽ tốt hơn")
print(f"\n   ✓ Sẽ train cả UserCF, ItemCF và SVD để so sánh")

PHÂN TÍCH CHI TIẾT DỮ LIỆU TRƯỚC TRAIN

1. THỐNG KÊ CƠ BẢN:
   - Số users: 610
   - Số movies: 9,720
   - Số ratings: 100,834
   - Tổng có thể ratings: 5,929,200
   - Sparsity (độ thưa): 0.9830 (98.30%)
   - Density (độ mật): 0.0170 (1.70%)

2. PHÂN BỐ RATING:
   - Min rating: 0.5
   - Max rating: 5.0
   - Mean rating: 3.50
   - Std rating: 1.04

   Hệ thống rating:
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26816
4.5     8551
5.0    13211
Name: count, dtype: int64

3. HOẠT ĐỘNG CỦA USER:
   - Ratings/user trung bình: 165.30
   - Min ratings/user: 20
   - Max ratings/user: 2698
   - Median ratings/user: 70

4. PHỔ BIẾN CỦA MOVIE:
   - Ratings/movie trung bình: 10.37
   - Min ratings/movie: 1
   - Max ratings/movie: 329
   - Median ratings/movie: 3

5. LỰA CHỌN THUẬT TOÁN DỰA VÀO DỮ LIỆU:
   • Sparsity cao (98.3%) → Cần xử lý sparsity tốt
   • User activity tương đối đều (avg=165.3) → UserCF/ItemCF khả thi
   • Movie populari

## 2) Chuẩn bị rating matrix (sparse)

Ở đây ta sẽ tạo train/test split dựa trên timestamp (giữ rating cuối cùng của mỗi user làm test) để đánh giá mô hình gợi ý theo thời gian.

In [ ]:
# Nếu chưa có timestamp, ta sẽ split random (vẫn đảm bảo user có train/test)
if 'timestamp' not in df_ratings.columns:
    df_ratings['timestamp'] = np.arange(len(df_ratings))

# 1) Chuyển sang kiểu số nếu cần
df_ratings['userId'] = df_ratings['userId'].astype(int)
df_ratings['movieId'] = df_ratings['movieId'].astype(int)
df_ratings['rating'] = df_ratings['rating'].astype(float)

# 2) Chia train/test: dùng phần tử mới nhất (timestamp lớn nhất) mỗi user làm test
df_ratings_sorted = df_ratings.sort_values(['userId', 'timestamp'])
test_idx = df_ratings_sorted.groupby('userId').tail(1).index
train_idx = df_ratings_sorted.index.difference(test_idx)

df_train = df_ratings.loc[train_idx].reset_index(drop=True)
df_test = df_ratings.loc[test_idx].reset_index(drop=True)

print('Train size:', len(df_train))
print('Test size:', len(df_test))

# Số lượng user và movie trong train/test
print('unique users train:', df_train['userId'].nunique())
print('unique users test:', df_test['userId'].nunique())
print('unique movies train:', df_train['movieId'].nunique())
print('unique movies test:', df_test['movieId'].nunique())


Train size: 100224
Test size: 610
unique users train: 610
unique users test: 610
unique movies train: 9697
unique movies test: 514


## 3) Thuật toán 1: User-based Collaborative Filtering (UserCF)

**Tại sao chọn**: UserCF dựa trên giả định rằng người dùng có hành vi tương đồng sẽ thích các item tương tự nhau. Phù hợp khi chúng ta có nhiều ratings từ từng user.

**Giải thích ngắn**:
- Tính similarity giữa các user dựa trên vector rating (cosine).
- Với một user cần dự đoán rating cho một phim, lấy các user tương đồng đã đánh giá phim đó và tính trung bình có trọng số.

Chúng ta sẽ đánh giá bằng RMSE/MAE trên tập test.

In [ ]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# Build mappings (userid/movieid -> index)
user_ids = df_train['userId'].unique()
movie_ids = df_train['movieId'].unique()
user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}

# Build sparse user-item matrix for train
rows = df_train['userId'].map(user_to_idx)
cols = df_train['movieId'].map(movie_to_idx)
data = df_train['rating']
R = csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(movie_ids)))

# Compute cosine similarity between users
user_sim = cosine_similarity(R)

def predict_usercf(user_id, movie_id, k=20):
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return np.nan
    uidx = user_to_idx[user_id]
    midx = movie_to_idx[movie_id]

    sims = user_sim[uidx]
    ratings = R[:, midx].toarray().reshape(-1)

    mask = ratings > 0
    if mask.sum() == 0:
        return np.nan

    sims = sims[mask]
    ratings = ratings[mask]

    top_k = np.argsort(sims)[-k:]
    sim_k = sims[top_k]
    rating_k = ratings[top_k]

    if sim_k.sum() == 0:
        return np.nan

    return np.dot(sim_k, rating_k) / sim_k.sum()

# Predict in bulk for test set (sample for speed)
df_test_sample = df_test.copy()
if len(df_test_sample) > 5000:
    df_test_sample = df_test_sample.sample(5000, random_state=42)

df_test_sample['pred_usercf'] = df_test_sample.apply(
    lambda r: predict_usercf(r['userId'], r['movieId'], k=20), axis=1
)

# Filter out cases where prediction couldn't be made
valid_mask = df_test_sample['pred_usercf'].notna()
rmse_usercf = np.sqrt(mean_squared_error(df_test_sample.loc[valid_mask, 'rating'], df_test_sample.loc[valid_mask, 'pred_usercf']))
mae_usercf = mean_absolute_error(df_test_sample.loc[valid_mask, 'rating'], df_test_sample.loc[valid_mask, 'pred_usercf'])

print(f'UserCF RMSE: {rmse_usercf:.4f}, MAE: {mae_usercf:.4f} (n={valid_mask.sum()} valid predictions)')

UserCF RMSE: 1.0473, MAE: 0.8281 (n=587 valid predictions)


## 4) Thuật toán 2: Item-based Collaborative Filtering (ItemCF)

**Tại sao chọn**: ItemCF ổn định hơn với user mới và ít thay đổi. Nó dựa vào ý tưởng "nếu bạn thích item A, bạn cũng có thể thích item B" dựa trên tính tương đồng giữa item.

**Giải thích ngắn**:
- Tính similarity giữa item dựa trên vector rating của user (cosine).
- Dự đoán rating là trung bình có trọng số các rating của user với các item tương đồng.


In [ ]:
# ItemCF: sử dụng ma trận user-item R đã tạo ở trên
from sklearn.metrics.pairwise import cosine_similarity

R_item = R.T.tocsr()  # shape: (n_items, n_users)
item_sim = cosine_similarity(R_item)

def predict_itemcf(user_id, movie_id, k=20):
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return np.nan
    uidx = user_to_idx[user_id]
    midx = movie_to_idx[movie_id]

    user_ratings = R[uidx].toarray().reshape(-1)
    mask = user_ratings > 0

    if mask.sum() == 0:
        return np.nan

    sims = item_sim[midx, mask]
    rating_k = user_ratings[mask]

    top_k = np.argsort(sims)[-k:]
    sim_k = sims[top_k]
    rating_k = rating_k[top_k]

    if sim_k.sum() == 0:
        return np.nan

    return np.dot(sim_k, rating_k) / sim_k.sum()

df_test_sample['pred_itemcf'] = df_test_sample.apply(
    lambda r: predict_itemcf(r['userId'], r['movieId'], k=20), axis=1
)

# Filter out cases where prediction couldn't be made
valid_mask = df_test_sample['pred_itemcf'].notna()
rmse_itemcf = np.sqrt(mean_squared_error(df_test_sample.loc[valid_mask, 'rating'], df_test_sample.loc[valid_mask, 'pred_itemcf']))
mae_itemcf = mean_absolute_error(df_test_sample.loc[valid_mask, 'rating'], df_test_sample.loc[valid_mask, 'pred_itemcf'])

print(f'ItemCF RMSE: {rmse_itemcf:.4f}, MAE: {mae_itemcf:.4f} (n={valid_mask.sum()} valid predictions)')

ItemCF RMSE: 0.9499, MAE: 0.7146 (n=587 valid predictions)


## 5) Thuật toán 3: Matrix Factorization (SVD)

**Tại sao chọn**: SVD (matrix factorization) học được embedding ẩn (latent factors) cho user và item, giúp xử lý sparsity tốt và cho dự đoán chính xác hơn.

**Giải thích ngắn**:
- Phân rã ma trận user-item thành hai ma trận thấp chiều (user factors, item factors)
- Dự đoán rating = dot(user_factor, item_factor).
- Huấn luyện bằng tối ưu hoá (SGD, ALS) trên dữ liệu rating.


In [ ]:
# Matrix Factorization via Surprise SVD
from surprise import SVD, Dataset, Reader
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Load data into Surprise
# Reader expects rating_scale to match the data min/max rating
reader = Reader(rating_scale=(df_ratings['rating'].min(), df_ratings['rating'].max()))

# Prepare dataset for Surprise
# IMPORTANT: Surprise expects df to have columns: user_id, item_id, rating
data = Dataset.load_from_df(df_train[['userId', 'movieId', 'rating']], reader)

# Train SVD
trainset = data.build_full_trainset()
svd = SVD(n_factors=50, random_state=42)
svd.fit(trainset)

# Evaluate on test set
preds = []
truths = []
for _, row in df_test_sample.iterrows():
    user_id = row['userId']
    movie_id = row['movieId']
    rating = row['rating']
    
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        continue
        
    est = svd.predict(user_id, movie_id).est
    preds.append(est)
    truths.append(rating)

rmse_svd = np.sqrt(mean_squared_error(truths, preds))
mae_svd = mean_absolute_error(truths, preds)
print(f'SVD (Surprise) RMSE: {rmse_svd:.4f}, MAE: {mae_svd:.4f} (n={len(truths)})')

SVD (Surprise) RMSE: 0.9651, MAE: 0.7422 (n=587)


## 6) Thuật toán 4: Content-Based Recommendation

**Tại sao chọn**: Content-Based dựa vào đặc trưng của item (phim) thay vì tương tác giữa user-item. Phương pháp này tốt cho cold-start problem (user hoặc phim mới) vì chỉ cần có đặc trưng của phim.

**Giải thích ngắn**:
- Xây dựng profile mỗi user dựa trên trung bình vector đặc trưng của các phim user đã đánh giá.
- Tính similarity giữa user profile và vector đặc trưng phim sử dụng cosine similarity.
- Dự đoán rating = cosine similarity * 5.0 (scale về [0, 5]).


In [ ]:
# --- Content-Based Recommendation ---
print("\n==== Content-Based Model ====")

# BƯỚC 1: Chuẩn bị vector đặc trưng cho mỗi phim
# Ghép movieId vào df_features (nếu chưa có)
if 'movieId' not in df_features.columns:
    df_features['movieId'] = df_metadata['movieId'].values

# Lấy các feature columns (bỏ qua text fields như title, tag, genres...)
content_features = df_features.set_index('movieId')
feature_cols = [c for c in content_features.columns if c not in ['userId','title','tag','genres','movie_era','runtime_category']]

# Chuẩn hóa đặc trưng theo cosine (để tính cosine similarity sau này)
from sklearn.preprocessing import normalize
X_content = content_features[feature_cols].fillna(0).values
X_content_norm = normalize(X_content, axis=1)  # L2 normalize each row
movie_feature_vector = {mid: X_content_norm[i] for i, mid in enumerate(content_features.index)}

# BƯỚC 2: Xây dựng user profile từ các phim user đã đánh giá trong training set
# User profile = trung bình vector đặc trưng của các phim mà user đó đã đánh giá
user_content_profiles = {}
for uid in df_train['userId'].unique():
    rated_movie_ids = df_train.loc[df_train['userId']==uid, 'movieId'].values
    vecs = [movie_feature_vector[m] for m in rated_movie_ids if m in movie_feature_vector]
    if len(vecs) == 0:
        # User chưa có rating hoặc phim không có đặc trưng -> profile = None
        user_content_profiles[uid] = None
    else:
        # Trung bình các vector đặc trưng (weighted)
        user_content_profiles[uid] = np.mean(vecs, axis=0)

from sklearn.metrics.pairwise import cosine_similarity as cossim

# BƯỚC 3: Hàm dự đoán rating dựa trên Content-Based
def predict_contentbased(user_id, movie_id):
    """
    Dự đoán rating = cosine similarity giữa user profile và movie feature vector
    Scale similarity về range [0.0, 5.0]
    """
    # Check user và movie có trong dữ liệu không
    if user_id not in user_content_profiles or movie_id not in movie_feature_vector:
        return np.nan
    
    profile = user_content_profiles[user_id]
    if profile is None:
        # User chưa có profile (không có rating trong train set)
        return np.nan
    
    # Tính cosine similarity
    sim = cossim(profile.reshape(1, -1), movie_feature_vector[movie_id].reshape(1, -1))[0, 0]
    
    # Clip vào range [0.0, 5.0] (scale rating)
    return np.clip(sim * 5.0, 0.0, 5.0)

# BƯỚC 4: Đánh giá Content-Based trên test set
print("\nEvaluating Content-Based on test set...")
p_preds = []
p_trues = []
for _, row in df_test.iterrows():
    pred = predict_contentbased(row['userId'], row['movieId'])
    if np.isnan(pred):
        continue
    p_preds.append(pred)
    p_trues.append(row['rating'])

# Tính RMSE và MAE
if len(p_preds) > 0:
    rmse_content = np.sqrt(mean_squared_error(p_trues, p_preds))
    mae_content = mean_absolute_error(p_trues, p_preds)
else:
    rmse_content = np.nan
    mae_content = np.nan

print(f"Content-Based RMSE: {rmse_content:.4f} | MAE: {mae_content:.4f} | n={len(p_preds)}")



==== Content-Based Model ====

Evaluating Content-Based on test set...
Content-Based RMSE: 1.7209 | MAE: 1.3213 | n=610


## 7) Đánh giá mô hình (Precision@K / Recall@K)

Để đánh giá gợi ý (top-K), ta sẽ tính `Precision@K` và `Recall@K` dựa trên việc gợi ý phim cho mỗi user và so sánh với các phim user thực sự đã đánh giá trong tập test.

In [ ]:
print("=" * 70)
print("PHÂN TÍCH OVERFITTING & DATA LEAKAGE")
print("=" * 70)

# Kiểm tra Train/Test performance gap
train_predictions = []
train_true = []

# Tính RMSE/MAE trên Train set (full training data)
for user_id in df_train['userId'].unique()[:100]:  # Sample 100 users for speed
    user_movies = df_train[df_train['userId'] == user_id]['movieId'].values
    for movie_id in user_movies[:5]:  # Sample 5 movies per user
        pred = predict_itemcf(user_id, movie_id, k=20)
        if not np.isnan(pred):
            actual = df_train[(df_train['userId'] == user_id) & 
                             (df_train['movieId'] == movie_id)]['rating'].values
            if len(actual) > 0:
                train_predictions.append(pred)
                train_true.append(actual[0])

if len(train_predictions) > 0:
    train_rmse = np.sqrt(mean_squared_error(train_true, train_predictions))
    train_mae = mean_absolute_error(train_true, train_predictions)
    print(f"\nItemCF Performance:")
    print(f"   Train RMSE: {train_rmse:.4f}")
    print(f"   Test RMSE:  {rmse_itemcf:.4f}")
    print(f"   Overfitting Gap: {(rmse_itemcf - train_rmse):.4f}")
    print(f"   → {'✓ OK (Gap < 0.2)' if (rmse_itemcf - train_rmse) < 0.2 else '⚠️ WARNING (Gap > 0.2)'}")

# Kiểm tra Content-Based overfitting
content_train_predictions = []
content_train_true = []
for user_id in df_train['userId'].unique()[:100]:
    user_movies = df_train[df_train['userId'] == user_id]['movieId'].values
    for movie_id in user_movies[:5]:
        pred = predict_contentbased(user_id, movie_id)
        if not np.isnan(pred):
            actual = df_train[(df_train['userId'] == user_id) & 
                             (df_train['movieId'] == movie_id)]['rating'].values
            if len(actual) > 0:
                content_train_predictions.append(pred)
                content_train_true.append(actual[0])

if len(content_train_predictions) > 0:
    content_train_rmse = np.sqrt(mean_squared_error(content_train_true, content_train_predictions))
    content_train_mae = mean_absolute_error(content_train_true, content_train_predictions)
    print(f"\nContent-Based Performance:")
    print(f"   Train RMSE: {content_train_rmse:.4f}")
    print(f"   Test RMSE:  {rmse_content:.4f}")
    print(f"   Overfitting Gap: {(rmse_content - content_train_rmse):.4f}")
    print(f"   → {'✓ OK (Gap < 0.2)' if (rmse_content - content_train_rmse) < 0.2 else '⚠️ WARNING (Gap > 0.2)'}")

# Kiểm tra Data Leakage
print(f"\nData Leakage Check:")
test_users_in_train = set(df_train['userId'].unique()) & set(df_test['userId'].unique())
print(f"   • Train users: {len(df_train['userId'].unique())}")
print(f"   • Test users: {len(df_test['userId'].unique())}")
print(f"   • Intersection: {len(test_users_in_train)}")
print(f"   → {'✓ No Leakage (OK)' if len(test_users_in_train) == len(df_test['userId'].unique()) else '⚠️ WARNING'}")

# Kiểm tra xem test phim/user có trong train không
test_movies_in_train = set(df_test['movieId'].unique()) & set(df_train['movieId'].unique())
print(f"\nTest Movies Check:")
print(f"   • Test unique movies: {len(df_test['movieId'].unique())}")
print(f"   • Test movies in train: {len(test_movies_in_train)}")
print(f"   • Cold-start movies: {len(df_test['movieId'].unique()) - len(test_movies_in_train)}")

print(f"\nPrecision/Recall Analysis:")
print(f"   • UserCF Precision@10: (will calculate below)")
print(f"   • ItemCF Precision@10: (will calculate below)")
print(f"   • SVD Precision@10: (will calculate below)")
print(f"   • Content-Based Precision@10: (will calculate below)")


⚠️  PHÂN TÍCH OVERFITTING & DATA LEAKAGE

📊 ItemCF Performance:
   Train RMSE: 0.7598
   Test RMSE:  0.9499
   Overfitting Gap: 0.1900
   → ✓ OK (Gap < 0.2)

📊 Content-Based Performance:
   Train RMSE: 1.7268
   Test RMSE:  1.7209
   Overfitting Gap: -0.0060
   → ✓ OK (Gap < 0.2)

🔍 Data Leakage Check:
   • Train users: 610
   • Test users: 610
   • Intersection: 610
   → ✓ No Leakage (OK)

📽️  Test Movies Check:
   • Test unique movies: 514
   • Test movies in train: 491
   • Cold-start movies: 23

🎯 Precision/Recall Analysis:
   • UserCF Precision@10: (will calculate below)
   • ItemCF Precision@10: (will calculate below)
   • SVD Precision@10: (will calculate below)
   • Content-Based Precision@10: (will calculate below)


## 8) ⚠️ Phân Tích Overfitting & Data Leakage

Kiểm tra xem các model có bị overfitting không bằng cách so sánh metrics trên Train vs Test

In [ ]:
def precision_recall_at_k(recommendations, ground_truth, k=10):
    precisions = []
    recalls = []
    for u, gt in ground_truth.items():
        if len(gt) == 0:
            continue
        recs = recommendations.get(u, [])[:k]
        hits = len(set(recs) & gt)
        precisions.append(hits / k)
        recalls.append(hits / len(gt))
    return np.mean(precisions) if precisions else np.nan, np.mean(recalls) if recalls else np.nan

gt_by_user = df_test.groupby('userId')['movieId'].apply(set).to_dict()
all_movies = set(df_train['movieId'].unique())

def recommend_usercf(user_id, k=10):
    scores = []
    for m in all_movies:
        if m in df_train[df_train['userId'] == user_id]['movieId'].values:
            continue
        score = predict_usercf(user_id, m, k=20)
        if not np.isnan(score):
            scores.append((m, score))
    top = [m for m, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]]
    return top

def recommend_itemcf(user_id, k=10):
    scores = []
    for m in all_movies:
        if m in df_train[df_train['userId'] == user_id]['movieId'].values:
            continue
        score = predict_itemcf(user_id, m, k=20)
        if not np.isnan(score):
            scores.append((m, score))
    top = [m for m, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]]
    return top

def recommend_svd(user_id, k=10):
    if user_id not in user_to_idx:
        return []
    user_rated = set(df_train.loc[df_train['userId'] == user_id, 'movieId'].values)

    scores = []
    for m in all_movies:
        if m in user_rated:
            continue
        if m not in movie_to_idx:
            continue
        est = svd.predict(user_id, m).est
        scores.append((m, est))

    top = [m for m, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]]
    return top
def recommend_contentbased(user_id, k=10):
    """Content-Based recommendation - gợi ý phim dựa trên similarity đặc trưng"""
    if user_id not in user_content_profiles:
        return []
    profile = user_content_profiles[user_id]
    if profile is None:
        return []
    
    scores = []
    user_rated = set(df_train.loc[df_train['userId'] == user_id, 'movieId'].values)
    
    for m in all_movies:
        if m in user_rated:
            continue
        if m not in movie_feature_vector:
            continue
        score = predict_contentbased(user_id, m)
        if not np.isnan(score):
            scores.append((m, score))
    
    top = [m for m, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:k]]
    return top

# Tính recommendation cho mỗi user
sample_users = list(gt_by_user.keys())[:200]
rec_usercf = {u: recommend_usercf(u, k=10) for u in sample_users}
rec_itemcf = {u: recommend_itemcf(u, k=10) for u in sample_users}
rec_svd = {u: recommend_svd(u, k=10) for u in sample_users}
rec_contentbased = {u: recommend_contentbased(u, k=10) for u in sample_users}

# Tính Precision@10 và Recall@10 cho mỗi model
p_u, r_u = precision_recall_at_k(rec_usercf, gt_by_user, k=10)
p_i, r_i = precision_recall_at_k(rec_itemcf, gt_by_user, k=10)
p_s, r_s = precision_recall_at_k(rec_svd, gt_by_user, k=10)
p_c, r_c = precision_recall_at_k(rec_contentbased, gt_by_user, k=10)

print('\n' + '='*70)
print('PRECISION & RECALL ANALYSIS @K=10')
print('='*70)
print(f'UserCF Precision@10: {p_u:.4f}, Recall@10: {r_u:.4f}')
print(f'ItemCF Precision@10: {p_i:.4f}, Recall@10: {r_i:.4f}')
print(f'SVD Precision@10: {p_s:.4f}, Recall@10: {r_s:.4f}')
print(f'Content-Based Precision@10: {p_c:.4f}, Recall@10: {r_c:.4f}')



PRECISION & RECALL ANALYSIS @K=10
UserCF Precision@10: 0.0000, Recall@10: 0.0000
ItemCF Precision@10: 0.0002, Recall@10: 0.0016
SVD Precision@10: 0.0007, Recall@10: 0.0066
Content-Based Precision@10: 0.0002, Recall@10: 0.0016


## 9) So sánh và kết luận

Tổng hợp kết quả RMSE/MAE và Precision@K/Recall@K để so sánh hiệu năng của 4 phương pháp.

**Phân tích từng thuật toán:**
- **UserCF**: Dựa trên user similarity, nhanh nhưng kém ổn định khi sparsity cao.
- **ItemCF**: Ổn định hơn UserCF, tốt cho recommendation khi user profiles không thay đổi nhiều.
- **SVD**: Hoạt động tốt với dữ liệu thưa, học được latent factors, cho dự đoán chính xác hơn.
- **Content-Based**: Không bị ảnh hưởng bởi data sparsity, phù hợp cho cold-start problem, nhưng phụ thuộc vào chất lượng của feature/metadata.

(Bạn có thể thay đổi tham số, số lượng factors, hoặc dùng LightFM / deep learning / hybrid models để cải thiện tiếp.)

**Lựa chọn thuật toán:**

- Nếu RMSE thấp nhất + Precision/Recall tốt -> Thuật toán phù hợp nhất.- Có thể kết hợp nhiều thuật toán (ensemble) để cải thiện hiệu năng.
- Nếu content features tốt -> Sử dụng Content-Based để giải quyết cold-start.

In [ ]:
import pickle
import joblib
from scipy.sparse import csr_matrix

# Lưu các model 
print("=" * 60)
print("LƯU CÁC MODEL ĐÃ TRAIN - OPTIMIZED")
print("=" * 60)

# 1. Lưu UserCF model (TRIM: chỉ giữ core data)
usercf_model = {
    'user_sim': user_sim,
    'R': R
} # <--- Thêm ngoặc đóng ở đây
joblib.dump(usercf_model, 'models/usercf_model.pkl')
print("✓ Lưu UserCF model -> models/usercf_model.pkl")
print(f"  • user_sim shape: {user_sim.shape}")
print(f"  • R shape: {R.shape}, sparsity: {1 - R.nnz / (R.shape[0] * R.shape[1]):.2%}")

# Lưu df_train riêng dưới dạng CSV
df_train.to_csv('models/df_train.csv', index=False)
print("✓ Lưu df_train -> models/df_train.csv")

# 2. Lưu ItemCF model (TRIM: chỉ giữ core data)
itemcf_model = {
    'item_sim': item_sim,
    'R': R
} # <--- Thêm ngoặc đóng ở đây
joblib.dump(itemcf_model, 'models/itemcf_model.pkl')
print("✓ Lưu ItemCF model -> models/itemcf_model.pkl")
print(f"  • item_sim shape: {item_sim.shape}")

# 3. Lưu SVD model (TRIM: chỉ giữ core data)
joblib.dump(svd, 'models/svd_model.pkl')
print("✓ Lưu SVD model -> models/svd_model.pkl")
print(f"  • User latent factors (pu) shape: {svd.pu.shape}")
print(f"  • Item latent factors (qi) shape: {svd.qi.shape}")

# 4. Lưu Content-Based model
# movie_feature_vector: dict với key=movieId, value=sparse vector
print("\nTối ưu Content-Based model...")

# Convert movie_feature_vector (dict of dense arrays) to dict of sparse vectors
movie_vec_sparse = {}
for movie_id, vec in movie_feature_vector.items():
    # Convert mỗi vector thành sparse CSR format (1, n_features) 
    movie_vec_sparse[movie_id] = csr_matrix(vec.reshape(1, -1))

print(f"  • Movie vectors converted to sparse format")
print(f"  • Sample: {list(movie_vec_sparse.keys())[:3]}")
print(f"  • Shape of first vector: {movie_vec_sparse[list(movie_vec_sparse.keys())[0]].shape}")

content_model = {
    'user_profile': user_content_profiles,
    'movie_vec': movie_vec_sparse  # Dict of sparse CSR matrices
} # <--- Thêm ngoặc đóng ở đây
joblib.dump(content_model, 'models/content_model.pkl')
print("✓ Lưu Content-Based model -> models/content_model.pkl (Sparse format)")
print(f"  • Số user profiles: {len(user_content_profiles)}")
print(f"  • Số movie vectors (sparse): {len(movie_vec_sparse)}")

# Calculate Content-Based metrics for consistency
try:
    p_c, r_c = precision_recall_at_k({u: [] for u in sample_users}, gt_by_user, k=10)
    if np.isnan(p_c):
        p_c = 0.0
    if np.isnan(r_c):
        r_c = 0.0
except:
    p_c = 0.0
    r_c = 0.0

# ============================================================
# 5. LƯU METADATA - CENTRALIZED INDEXING
# Chứa tất cả mappings (user_to_idx, movie_to_idx, v.v.)
# để tránh duplicating khi load multiple models
# ============================================================
metadata = {
    # === INDEXING (CENTRALIZED) ===
    'user_to_idx': user_to_idx,
    'idx_to_user': {i: u for u, i in user_to_idx.items()},
    'movie_to_idx': movie_to_idx,
    'idx_to_movie': {i: m for m, i in movie_to_idx.items()},
    'user_ids': user_ids,
    'movie_ids': movie_ids,
    
    # === PERFORMANCE METRICS ===
    'rmse_scores': {
        'usercf': rmse_usercf,
        'itemcf': rmse_itemcf,
        'svd': rmse_svd,
        'content': rmse_content if not np.isnan(rmse_content) else 0.0
    },
    'mae_scores': {
        'usercf': mae_usercf,
        'itemcf': mae_itemcf,
        'svd': mae_svd,
        'content': mae_content if not np.isnan(mae_content) else 0.0
    },
    'precision_at_10': {
        'usercf': p_u,
        'itemcf': p_i,
        'svd': p_s,
        'content': p_c
    },
    'recall_at_10': {
        'usercf': r_u,
        'itemcf': r_i,
        'svd': r_s,
        'content': r_c
    },
    
    # === NOTES ===
    'notes': {
        'optimization': 'Indexing được centralize trong metadata để tránh duplicate khi load multiple models',
        'content_model': 'Movie features được lưu ở dạng sparse CSR matrix để tiết kiệm RAM',
        'all_models_load': 'Load metadata.pkl một lần duy nhất, sau đó load từng model phù hợp'
    }
} # <--- Thêm ngoặc đóng ở đây
joblib.dump(metadata, 'models/model_metadata.pkl')
print("✓ Lưu metadata -> models/model_metadata.pkl")
print(f"  • user_to_idx: {len(metadata['user_to_idx'])} users")
print(f"  • movie_to_idx: {len(metadata['movie_to_idx'])} movies")
print(f"  • Metrics: RMSE, MAE, Precision@10, Recall@10 cho 4 models")

# 5. Lưu df_metadata cho thông tin phim
df_metadata.to_csv('models/movies_metadata_for_testing.csv', index=False)
print("✓ Lưu movie metadata -> models/movies_metadata_for_testing.csv")

print("\n" + "=" * 70)
print("TÓM TẮT HIỆU NĂNG CÁC MODEL")
print("=" * 70)
print("\nMetrics trên tập test:")
print(f"┌──────────────┬──────────┬──────────┬───────────┬──────────┐")
print(f"│   Model      │   RMSE   │   MAE    │ Prec@10   │ Recall@10│")
print(f"├──────────────┼──────────┼──────────┼───────────┼──────────┤")
print(f"│   UserCF     │  {rmse_usercf:.4f}  │  {mae_usercf:.4f}  │  {p_u:.4f}   │  {r_u:.4f}  │")
print(f"│   ItemCF     │  {rmse_itemcf:.4f}  │  {mae_itemcf:.4f}  │  {p_i:.4f}   │  {r_i:.4f}  │")
print(f"│    SVD       │  {rmse_svd:.4f}  │  {mae_svd:.4f}  │  {p_s:.4f}   │  {r_s:.4f}  │")
print(f"│ Content-Based│  {rmse_content:.4f}  │  {mae_content:.4f}  │  {p_c:.4f}   │  {r_c:.4f}  │")
print(f"└──────────────┴──────────┴──────────┴───────────┴──────────┘")

print("\n" + "=" * 70)
print("⚡ TỐI ƯU HÓA - MEMORY EFFICIENCY")
print("=" * 70)

print("\nFILES SAVED:")
print("  ✓ models/usercf_model.pkl")
print("  ✓ models/itemcf_model.pkl")
print("  ✓ models/svd_model.pkl")
print("  ✓ models/content_model.pkl (movie features in sparse format)")
print("  ✓ models/model_metadata.pkl (centeralized indexing + metrics)")
print("  ✓ models/movies_metadata_for_testing.csv")
print("  ✓ models/df_train.csv")

print("\nCác model sẵn sàng để test trong Test_Model.ipynb hoặc app.py!")


LƯU CÁC MODEL ĐÃ TRAIN - OPTIMIZED
✓ Lưu UserCF model -> models/usercf_model.pkl
  • user_sim shape: (610, 610)
  • R shape: (610, 9697), sparsity: 98.31%
✓ Lưu df_train -> models/df_train.csv
✓ Lưu ItemCF model -> models/itemcf_model.pkl
  • item_sim shape: (9697, 9697)
✓ Lưu SVD model -> models/svd_model.pkl
  • User latent factors (pu) shape: (610, 50)
  • Item latent factors (qi) shape: (9697, 50)

⚙️ Tối ưu Content-Based model...
  • Movie vectors converted to sparse format
  • Sample: [1, 3, 6]
  • Shape of first vector: (1, 128)
✓ Lưu Content-Based model -> models/content_model.pkl (Sparse format)
  • Số user profiles: 610
  • Số movie vectors (sparse): 9720
✓ Lưu metadata -> models/model_metadata.pkl
  • user_to_idx: 610 users
  • movie_to_idx: 9697 movies
  • Metrics: RMSE, MAE, Precision@10, Recall@10 cho 4 models
✓ Lưu movie metadata -> models/movies_metadata_for_testing.csv

📊 TÓM TẮT HIỆU NĂNG CÁC MODEL

Metrics trên tập test:
┌──────────────┬──────────┬──────────┬────────

## 10) Lưu các model đã train

Lưu các model và mapping để sử dụng trong phần test/inference. Sẽ lưu các file sau:
- `usercf_model.pkl`: User similarity matrix và mappings
- `itemcf_model.pkl`: Item similarity matrix và mappings  
- `svd_model.pkl`: SVD model (U_s, V components)
- `content_model.pkl`: Content-Based model (user profiles & movie feature vectors)
- `model_metadata.pkl`: User/Movie ID mappings và performance metrics